# French to Tamil Machine Learning model

##### `Problem Statement : French to Tamil: Description: Make a machine learning model with the feature that translates French words into Tamil. The feature should only translate French words that have exactly five letters. If a French word has more or fewer than five letters, the model should not translate it. Guidelines: You have to make a GUI for this task. The GUI should include an input section for entering French words and an output section for displaying the translated Tamil words.`

## Load all the important libraries

In [3]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
import tensorflow as tf
import json

import re


ModuleNotFoundError: No module named 'numpy'

## Load the dataset

In [3]:
data = pd.read_csv("french_tamil_5letter.csv") 
data

,French,Tamil
0,était,இருந்தது
1,faire,செய்ய
2,comme,என
3,votre,உங்கள்
4,cette,இது
...,...,...
982,poète,கவிஞர்
983,admis,ஒப்புக்கொண்டார்
984,drake,டிரேக்
985,lydia,லிடியா


## Preprocess the data

In [18]:
data['Tamil'] = data['Tamil'].apply(lambda x: '\t' + x + '\n')

french_tk = Tokenizer(char_level=True, filters='')
tamil_tk  = Tokenizer(char_level=True, filters='')

french_tk.fit_on_texts(data['French'])
tamil_tk.fit_on_texts(data['Tamil'])

french_seq = french_tk.texts_to_sequences(data['French'])
tamil_seq  = tamil_tk.texts_to_sequences(data['Tamil'])

max_french_len = max(len(s) for s in french_seq)
max_tamil_len  = max(len(s) for s in tamil_seq)

french_seq = pad_sequences(french_seq, maxlen=max_french_len, padding='post')
tamil_seq  = pad_sequences(tamil_seq,  maxlen=max_tamil_len,  padding='post')

X_train, X_val, y_train, y_val = train_test_split(
    french_seq, tamil_seq, test_size=0.2, random_state=42
)

## Create a model

In [28]:
french_vocab_size = len(french_tk.word_index) + 1
tamil_vocab_size  = len(tamil_tk.word_index) + 1
latent_dim = 256

In [30]:
encoder_inputs = Input(shape=(max_french_len,))
enc_emb = Embedding(french_vocab_size, 128, mask_zero=True)(encoder_inputs)
_, state_h, state_c = LSTM(latent_dim, return_state=True)(enc_emb)
encoder_states = [state_h, state_c]

decoder_inputs = Input(shape=(max_tamil_len - 1,))
dec_emb_layer = Embedding(tamil_vocab_size, 128, mask_zero=True)
dec_emb = dec_emb_layer(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

decoder_dense = Dense(tamil_vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)



In [32]:
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

Model: "model_8"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_10 (InputLayer)          [(None, 5)]          0           []                               
                                                                                                  
 input_11 (InputLayer)          [(None, 21)]         0           []                               
                                                                                                  
 embedding_5 (Embedding)        (None, 5, 128)       4736        ['input_10[0][0]']               
                                                                                                  
 embedding_6 (Embedding)        (None, 21, 128)      7168        ['input_11[0][0]']               
                                                                                            

In [33]:
decoder_input_train  = y_train[:, :-1]
decoder_target_train = np.expand_dims(y_train[:, 1:], -1)

decoder_input_val  = y_val[:, :-1]
decoder_target_val = np.expand_dims(y_val[:, 1:], -1)

## Train the model

In [34]:
model.fit(
    [X_train, decoder_input_train],
    decoder_target_train,
    validation_data=([X_val, decoder_input_val], decoder_target_val),
    batch_size=32,
    epochs=100,
    verbose=2
)

Epoch 1/100
25/25 - 14s - loss: 1.5138 - accuracy: 0.1828 - val_loss: 1.3237 - val_accuracy: 0.1770 - 14s/epoch - 568ms/step
Epoch 2/100
25/25 - 1s - loss: 1.2587 - accuracy: 0.2401 - val_loss: 1.2281 - val_accuracy: 0.2891 - 571ms/epoch - 23ms/step
Epoch 3/100
25/25 - 0s - loss: 1.1521 - accuracy: 0.3117 - val_loss: 1.1185 - val_accuracy: 0.3460 - 446ms/epoch - 18ms/step
Epoch 4/100
25/25 - 1s - loss: 1.0518 - accuracy: 0.3594 - val_loss: 1.0280 - val_accuracy: 0.3660 - 569ms/epoch - 23ms/step
Epoch 5/100
25/25 - 0s - loss: 0.9736 - accuracy: 0.3725 - val_loss: 0.9731 - val_accuracy: 0.3698 - 467ms/epoch - 19ms/step
Epoch 6/100
25/25 - 1s - loss: 0.9349 - accuracy: 0.3797 - val_loss: 0.9464 - val_accuracy: 0.3871 - 596ms/epoch - 24ms/step
Epoch 7/100
25/25 - 0s - loss: 0.9074 - accuracy: 0.3977 - val_loss: 0.9200 - val_accuracy: 0.4147 - 460ms/epoch - 18ms/step
Epoch 8/100
25/25 - 0s - loss: 0.8863 - accuracy: 0.4038 - val_loss: 0.9036 - val_accuracy: 0.4277 - 450ms/epoch - 18ms/step


In [35]:
encoder_model = Model(encoder_inputs, encoder_states)

decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

dec_emb2 = dec_emb_layer(decoder_inputs)
decoder_outputs2, h2, c2 = decoder_lstm(
    dec_emb2, initial_state=decoder_states_inputs
)
decoder_states2 = [h2, c2]
decoder_outputs2 = decoder_dense(decoder_outputs2)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs2] + decoder_states2
)


## Save the model

In [36]:
# Save main model (training model)
model.save("french_tamil_model.h5")

# Save encoder model
encoder_model.save("encoder_model.h5")

# Save decoder model
decoder_model.save("decoder_model.h5")

## Save the Tokenizers

In [37]:
#  Save tokenizers as JSON
from tensorflow.keras.preprocessing.text import tokenizer_from_json
import json

with open("french_tokenizer.json", "w", encoding="utf-8") as f:
    f.write(french_tk.to_json())

with open("tamil_tokenizer.json", "w", encoding="utf-8") as f:
    f.write(tamil_tk.to_json())

print("Tokenizers saved successfully!")

Tokenizers saved successfully!


## Prediction fucntion

In [ ]:
def clean_tamil_text(text):
    return text.replace('\t', '').replace('\n', '').strip()

def decode_sequence(input_seq):
    states_value = encoder_model.predict(input_seq, verbose=0)

    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = tamil_tk.word_index['\t']  

    decoded_sentence = ''

    while True:
        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value, verbose=0
        )

        sampled_index = np.argmax(output_tokens[0, -1, :])
        

        sampled_char = None
        for char, index in tamil_tk.word_index.items():
            if index == sampled_index:
                sampled_char = char
                break

        if sampled_char == '\n' or len(decoded_sentence) > max_tamil_len:
            break

        decoded_sentence += sampled_char

        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_index
        states_value = [h, c]

    return clean_tamil_text(decoded_sentence)

## Predict using the model

In [39]:
def translate_french_word(french_word):
    if len(french_word) != 5:
        return "Word must have exactly 5 letters"

    seq = french_tk.texts_to_sequences([french_word])
    seq = pad_sequences(seq, maxlen=max_french_len, padding='post')

    return decode_sequence(seq)

In [42]:
french_input = input("Enter a 5-letter French word: ").strip().lower()
tamil_output = translate_french_word(french_input)

print(f"\n🇫🇷 French: {french_input}")
print(f"🇮🇳 Tamil: {tamil_output}")

Enter a 5-letter French word:  gamin



🇫🇷 French: gamin
🇮🇳 Tamil: குழந்தை
